[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/wing/wing.ipynb)

# Potential Flow past a Joukowski Wing

This example computes the potential flow past a Joukowski wing and the lift it gives. In an incompressible, inviscid flow the velocity has no divergence, since the fluid cannot be compressed, and no vorticity in the fluid, since without viscosity none is ever made: the velocity's derivative vanishes at every point around the wing. And still the wing is lifted. The vorticity that lifts it is bound in the wing itself, and the flow outside sees it only as a circulation around any loop that encloses the wing, the same around every one. The flow is built past a cylinder and carried onto the wing by Joukowski's map, which keeps angles and so keeps the flow ideal.

Points, and the velocities that move them, are modelled as antivectors. The velocity's gradient is a map from a small step to the change in velocity, and multiplied by `Point`, the space of all points, and contracted, `(Point * gradient(Point)).contract(1, 2)` is the velocity's geometric derivative: its scalar part the divergence and its bivector part the vorticity, both zero. The circulation is a bivector, `4 * pi * (stream ^ edge)`, and the Joukowski map adds to each point its inverse sandwiched by a critical point, `p + critical * p.inverse() * critical`; its Jacobian turns and scales every step, which is what carries an ideal flow to an ideal flow.

In the notation of complex analysis this is the classical route of Joukowski: the complex potential of the flow past a cylinder, analytic by the equations of Cauchy and Riemann, carried to the wing by his map $z = \zeta + c^2/\zeta$, with the velocity the conjugate of its derivative and the lift given by the Kutta–Joukowski theorem.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from dataclasses import dataclass

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA2D as ga
from examples.util.animation import save_figure, show_animation
from examples.fluids.wing import core, render
from examples.fluids.wing.core import rings

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Points, and displacements and velocities of points, are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()
Bivector = ga.gatype.bivector()
VelocityGradient = ga.gatype((Point, Point))                               # Point <- Point
PotentialGradient = ga.gatype((Scalar, Point))                              # Scalar <- Point

## 1. The cylinder and its circulation

The wing starts as a cylinder. A map, built in section 3, will fold the cylinder's surface at one point into the wing's sharp trailing edge, and the flow can only leave that edge smoothly if it stands still there before the map. One circulation around the cylinder makes it so, the Kutta condition, and it comes out as a bivector: the stream wedged with the direction from the cylinder's centre to the trailing edge, `4 * pi * (stream ^ edge)`. The lift is the circulation contracted with the stream, times the density.

In scalar notation the lift reads as $L = \rho U \Gamma$, with $\Gamma = 4\pi U R \sin(\alpha + \beta)$ for a cylinder of radius $R$ whose trailing edge lies at the angle $\beta$ below its centre.

In [ ]:
# The wing level along the stream; the cylinder's centre before the map sets its thickness and camber. The
# stream's speed and density, and the angle of attack.
thickness, camber, scale = 0.1, 0.08, 1.0
speed, density = 1.0, 1.0
attack = np.deg2rad(8.0)

In [ ]:
@dataclass(frozen=True)
class Wing:
    centre: Point                                           # [] Point, the cylinder's centre before the map
    scale: float                                             # the cylinder passes through scale * chord, the trailing edge
    chord: Point                                            # [] Point, unit, from the leading to the trailing edge


# The wing pitched nose up by the angle of attack: one rotor turns the cylinder's centre and the chord,
# level along the stream, together, and the map built on the chord turns with them.
turn = (mv.xy * (attack / 2)).exp()                                            # [] Rotor
wing = Wing(turn >> (-thickness * mv.x + camber * mv.y).dual(), scale, turn >> mv.x.dual())   # [] Wing
stream = speed * mv.x.dual()                                                   # [] Point
# From the cylinder's centre to the trailing edge, where the flow must stand still.
edge = wing.scale * wing.chord - wing.centre                                   # [] Point
# The circulation that stops it there, and the lift it gives.
swirl = 4 * np.pi * (stream ^ edge)                                            # [] Bivector
lift = density * (swirl | stream)                                              # [] Point

## 2. The velocity past the cylinder

Past the cylinder the velocity is three terms: the stream itself; less its image in the cylinder, the stream sandwiched between two inverses of the position and scaled by the radius squared, which cancels the flow through the cylinder's surface; and the swirl, the circulation times the position's inverse. A small step changes the position's inverse by minus the step sandwiched between two inverses, and that gives the velocity's gradient map, `Point <- Point`. Multiplied by `Point`, the space of all points, and contracted, it is the velocity's derivative, and it vanishes everywhere: no divergence, no vorticity.

A step also changes two fields that describe the flow: the velocity potential by `velocity | step`, how far the step goes along the flow, and the stream function, a bivector, by `velocity ^ step`, how much flow crosses it. The two are the parts of one product, `velocity * step`. The stream function is constant along each streamline, and it is written out below.

In [ ]:
# The rings around the cylinder the flow is sampled on, the angles around each, and how far out they reach.
n_rings, n_angles, reach = 140, 360, 6.0

In [ ]:
def cylinder(centre: Point, edge: Point, swirl: Bivector, stream: Point, plane: Point) -> tuple[Point, VelocityGradient, Bivector]:
    """The flow past a cylinder through `centre + edge`: its velocity, the velocity's gradient map,
    and the stream function."""
    offsets = plane - centre                                                   # [...] Point
    radius_squared = edge | edge                                               # [] Scalar
    inverse = offsets.inverse()                                                # [...] Point
    # The stream, less its image in the cylinder, and the swirl around it.
    velocity = stream - radius_squared * (inverse * stream * inverse) - swirl * inverse / (2 * np.pi)   # [...] Point
    # A step changes the inverse by minus the step between two inverses, and the velocity with it.
    change = -(inverse * Point * inverse)                                     # [...] Point <- Point
    gradient = -radius_squared * (change * stream * inverse + inverse * stream * change) - swirl * change / (2 * np.pi)
    # The stream function: the flow crossing from the centre's line along the stream, the image's,
    # and the swirl's, which grows with the logarithm of the distance.
    flux = (stream ^ offsets) + radius_squared * (inverse ^ stream) - swirl * (offsets | offsets).log() / (4 * np.pi)
    return velocity, gradient, flux


# Points on rings around the cylinder, from its surface out.
plane = rings(wing, n_rings, n_angles, reach)                                # [rings, angles] Point
velocity, gradient, flux = cylinder(wing.centre, edge, swirl, stream, plane)
# The velocity's derivative: the gradient multiplied by `Point`, the space of all points, and contracted.
derivatives = (Point * gradient(Point)).contract(1, 2)                       # [rings, angles] Even
print("largest divergence or vorticity past the cylinder:", np.abs(derivatives.kernel).max())

## 3. The map

The Joukowski map turns the cylinder into the wing. It adds to each point its inverse sandwiched by a critical point, `p + critical * p.inverse() * critical`: reflected in the line along the critical point, and scaled by its squared distance. A small step moves under the map by its Jacobian: the step, less the step turned and scaled by the even multivector `critical * p.inverse()`. Turning and scaling keeps angles, so the map keeps the angle at which any two curves cross. The exceptions are `critical` and `-critical`, where `critical * p.inverse()` is one or minus one and the Jacobian vanishes; there the map doubles angles instead, and moves the two points to `2 * critical` and its negative.

The figure shows what the map does to two families of circles. Circles through both critical points land on circles through their images; circles around one critical point, the points at one ratio of distances to the two, land on circles around its image, at the ratio squared. The circle centred between the critical points crosses through both at right angles, which double to straight ones: it flattens onto the plate between the images. The wing's circle passes through `critical` and goes around `-critical`. Where it passes through, the doubled angle folds its smooth edge into a cusp, the sharp trailing edge.

In the notation of complex analysis the map reads as $z = \zeta + c^2/\zeta$, the circles as the bipolar coordinates of the two critical points, and the doubling as $\frac{z - 2c}{z + 2c} = \left(\frac{\zeta - c}{\zeta + c}\right)^2$.

In [ ]:
# The grid of circles drawn: centres of the circles through both critical points, across the chord in steps
# of the critical point's distance, and distance ratios of the circles around one of them; and the points
# along each circle.
through_offsets = np.array([-1.5, -0.6, 0.0, 0.6, 1.5])
distance_ratios = np.array([0.2, 0.45, 0.7])
n_outline_points = 400

In [ ]:
def joukowski(critical: Point, plane: Point) -> Point:
    """Each point plus its inverse sandwiched by the critical point: reflected in the line along it, and
    scaled by its squared distance."""
    return plane + critical * plane.inverse() * critical                       # [...] Point


# The trailing edge is a critical point: the wing's circle passes through it.
critical = wing.scale * wing.chord                                             # [] Point
turns = (mv.xy * np.linspace(0, np.pi, n_outline_points)).exp()                 # [outline_points] Rotor
# Circles through both critical points, centred on the line across the chord through the map's centre.
across = through_offsets * ((mv.xy * (-np.pi / 4)).exp() >> critical)          # [through] Point
reach_through = ((critical | critical) + (across | across)).square_root()      # [through] Scalar
through = across[:, None] + reach_through[:, None] * (turns >> wing.chord)[None, :]   # [through, outline_points] Point
# Circles around each critical point, the points at one ratio of distances to the two: centred
# along the chord, beyond the critical point they surround.
ratios = np.concatenate([distance_ratios, 1 / distance_ratios])
centres = critical * ((1 + ratios**2) / (1 - ratios**2))                       # [around] Point
reach_around = (critical | critical).square_root() * (2 * ratios / np.abs(1 - ratios**2))   # [around] Scalar
around = centres[:, None] + reach_around[:, None] * (turns >> wing.chord)[None, :]   # [around, outline_points] Point
grid = concatenate([through, around])                                          # [circles, outline_points] Point
wing_circle = wing.centre + (turns >> edge)                                    # [outline_points] Point
save_figure(render.bipolar(grid, joukowski(critical, grid), wing_circle, joukowski(critical, wing_circle), critical), "wing_map");

## 4. Carried to the wing

The flow at the wing is the cylinder's carried along: the velocity potential and the stream function take, at each point's image, the values they had at the point. That this is again an ideal flow is what keeping angles buys. The Jacobian turns and scales every step, so the carried velocity is the cylinder's, turned as the steps are and divided by the scale, and its divergence and vorticity are the cylinder's divided by the scale squared: still zero. Lines of one value of the stream function land on lines of one value, so the cylinder's surface, a streamline, lands on the wing's, and no flow crosses it; far away the map barely moves anything, and the flow is still the stream.

A step at the image is first undone by the Jacobian, so the potential's gradient map at the cylinder, `velocity | Point`, composed with the inverse of the Jacobian, `jacobian.inverse()`, is its gradient map at the wing. The velocity at the wing is its derivative, the same contraction with `Point`, and the stream function's values at the images draw the wing's streamlines. In the picture the pressure is low where the flow is fast, over the wing, and the arrow is the lift.

In [ ]:
def flow(wing: Wing, swirl: Bivector, stream: Point, plane: Point, critical: Point
         ) -> tuple[Point, Point, PotentialGradient, Bivector]:
    """The flow at the images of the points: the images, the velocity there, the potential's gradient
    map, and the stream function."""
    velocity, _, flux = cylinder(wing.centre, wing.scale * wing.chord - wing.centre, swirl, stream, plane)
    # How the map moves a small step: less the step turned and scaled by the even `critical * p.inverse()`.
    jacobian = Point - ((critical * plane.inverse()) >> Point)              # [...] Point <- Point
    # The potential keeps its value at a point's image: a step there is first undone by the Jacobian.
    potential_gradient = (velocity | Point)(jacobian.inverse())                # [...] Scalar <- Point
    # The velocity at the wing: the potential's derivative.
    at_wing = (Point * potential_gradient(Point)).contract(1, 2)            # [...] Point
    return joukowski(critical, plane), at_wing, potential_gradient, flux


wing_points, wing_velocity, _, wing_stream = flow(wing, swirl, stream, plane, critical)   # [rings, angles]
save_figure(render.figure(wing_points, wing_velocity, wing_stream, lift, speed), "wing_flow");

Moving the critical points apart deforms the cylinder into the wing. While they lie inside it, the map is smooth on its surface and the body is round; as `critical` reaches the surface, its doubled angle there turns into the cusp. The yellow points are their images, `2 * critical` and its negative. They start together at the centre of the map, while the cylinder's centre, the white point, sits behind it by the thickness and above it by the camber: that offset is all that makes the wing differ front to back and top to bottom. The circulation is held at the wing's throughout, so the last frame is the wing's flow.

In [ ]:
# The animation draws from fewer rings and angles, all a picture needs; its frames, and the duration of each.
n_drawn_rings, n_drawn_angles = 35, 90
n_morph_frames, duration_ms = 24, 80

In [ ]:
def morphing(n_frames: int):
    """Per frame, the flow as the critical points move apart from the map's centre out to the trailing edge,
    with the lift, the critical points' images and the cylinder's centre. The cylinder and its circulation
    stay put; only the map changes, the identity at first."""
    drawn = rings(wing, n_drawn_rings, n_drawn_angles, reach)                  # [rings, angles] Point
    for fraction in np.linspace(0.0, 1.0, n_frames):
        points, velocity, _, flux = flow(wing, swirl, stream, drawn, fraction * critical)
        # Marked: where the map sends the critical points, twice as far out, and the cylinder's centre.
        yield points, velocity, flux, lift, 2 * fraction * stack([critical, -critical]), wing.centre


show_animation(render.animate_marked(morphing(n_morph_frames), speed), "wing_morph", duration_ms)

## 5. Around the wing

Around a ring about the wing, the velocity times each step, summed, adds up the changes of both fields once around: the potential's, the scalar part, and the stream function's, the bivector part. The stream function comes back to where it started, so the bivector part is zero, no net flow out of the ring. The potential does not: it gains the circulation. Every ring gives the same sum, because between any two rings the velocity's derivative vanishes; the circulation is held by the loop around the wing, not by any point of the flow.

The pressure on the surface pushes the wing by half the density times the squared speed, along the inward normal at each step. Summed around the wing, that force is the lift from the circulation, across the stream, and nothing along it: an ideal flow that lifts but does not drag, d'Alembert's paradox.

In the notation of vector calculus the sum's two parts read as the circulation $\oint \mathbf v \cdot d\mathbf l$ and the flux $\oint \mathbf v \times d\mathbf l$, and their agreement over every ring as the theorems of Stokes and Gauss.

In [ ]:
# Around every ring, the velocity times each step, the two ends of the step averaged.
next_idx = (np.arange(n_angles) + 1) % n_angles
steps = wing_points[:, next_idx] - wing_points                               # [rings, angles] Point
loops = ((wing_velocity + wing_velocity[:, next_idx]) / 2 * steps).sum(axis=-1)   # [rings] Even
# The surface: the innermost ring. The pressure is low where the flow is fast: it pushes by half the
# density times the squared speed, averaged over each step, along the inward normal, the step turned
# a quarter turn back.
surface, surface_velocity, _, _ = flow(wing, swirl, stream, rings(wing, 1, n_angles, 1.0)[0], critical)   # [angles] Point each
surface_steps = surface[next_idx] - surface                      # [angles] Point
inward = (mv.xy * (np.pi / 4)).exp() >> surface_steps                          # [angles] Point
speed_squared = surface_velocity | surface_velocity                            # [angles] Scalar
force = (0.5 * density * (speed_squared + speed_squared[next_idx]) / 2 * inward).sum(axis=-1)   # [] Point

In [ ]:
print(f"around the nearest ring:\n{loops[0].coefficient_table()}")
print(f"around the farthest ring:\n{loops[-1].coefficient_table()}")
print(f"the circulation:\n{swirl.coefficient_table()}")
print(f"pressure force:\n{force.coefficient_table()}")
print(f"lift from the circulation:\n{lift.coefficient_table()}")

In [ ]:
# checks
# The notebook's flow is the core's.
_, reference_velocity, _, _ = core.flow(wing, stream, plane, critical)
np.testing.assert_allclose((reference_velocity - wing_velocity).kernel, 0.0, atol=1e-12)
# The map sends the critical points to twice themselves.
np.testing.assert_allclose((joukowski(critical, critical) - 2 * critical).kernel, 0.0, atol=1e-12)
# A circle around a critical point lands on a circle around its image, at its distance ratio squared.
landed = joukowski(critical, around)                                           # [around, outline_points] Point
distance_ratio = ((landed - 2 * critical).norm() / (landed + 2 * critical).norm()).to_array()
np.testing.assert_allclose(distance_ratio, (ratios**2)[:, None] * np.ones(n_outline_points), rtol=1e-9)
# The velocity has no derivative anywhere past the cylinder.
np.testing.assert_allclose(derivatives.kernel, 0.0, atol=1e-12)
# A step changes the stream function by the velocity's flux through it, at the wing as at the cylinder.
step = mv.vector([0.3, -0.2]).dual() * 1e-6                                    # [] Point
forward_points, _, _, forward_stream = flow(wing, swirl, stream, plane + step, critical)
backward_points, _, _, backward_stream = flow(wing, swirl, stream, plane - step, critical)
crossing = (forward_points - backward_points) / 2                              # [rings, angles] Point
np.testing.assert_allclose(((forward_stream - backward_stream) / 2 - (wing_velocity ^ crossing)).kernel, 0.0, atol=1e-12)
# Every ring gives the same sum, the circulation with no flux, and the pressure force is the lift,
# to the resolution of the rings.
np.testing.assert_allclose(loops.kernel - loops.kernel[:1], 0.0, atol=1e-11)
# Each ring's own oriented area sets the sense its sum is read in.
areas = (wing_points ^ wing_points[:, next_idx]).sum(axis=-1)                # [rings] Bivector
np.testing.assert_allclose((loops * (areas / areas.norm()) - swirl).kernel, 0.0, atol=1e-3)
np.testing.assert_allclose((force - lift).kernel, 0.0, atol=1e-3)